# Recorded predictions

Two ways to predict from a notebook, and the choice is about *bookkeeping*, not
speed:

| | `models.predictor()` | `inference.predict()` |
| --- | --- | --- |
| Runs in | this kernel | the API process |
| Cost | one model load, then fast | an HTTP round trip each |
| Leaves behind | nothing | an id, an overlay, a row on `/inference` |

Use the first for a whole split. Use this one when the answer is something you
will want to point at later.

In [ ]:
import orinth

MODEL_ID = next(model.id for model in orinth.models.list() if model.available)
model = orinth.models.get(MODEL_ID)
print(model)

In [ ]:
source = next(
    dataset
    for dataset in orinth.datasets.list()
    if dataset.task_type == model.task_type and dataset.splits.get("test")
)
paths = orinth.datasets.paths(source.id, "test")[:5]
print(source.id, "|", len(paths), "images to try")

## Predict

Parameters are the same ones the studio sends, and they mean the same thing.
Lowering `confidence_threshold` here changes this call only — it is not a
setting.

In [ ]:
predictions = [
    orinth.inference.predict(MODEL_ID, path, confidence_threshold=0.5)
    for path in paths
]

for prediction in predictions:
    top = sorted(prediction.scores.items(), key=lambda item: -item[1])[:2]
    print(f"{prediction.id[:12]}  {prediction.label:14} {top}  {len(prediction.detections)} detections")

## Look at one

`overlay_url` is served by the platform, so the same image is on the Inference
page. Reading it here is the whole point of recording rather than predicting in
the kernel: the artifact outlives the cell.

In [ ]:
from IPython.display import Image, display

first = predictions[0]
print(first.label, "|", first.overlay_url)
overlay = orinth.settings.workspace()["storage"] + first.overlay_url.replace("/media", "")
display(Image(filename=overlay))

## Text models take the same call

`text=` instead of a path. Exactly one of the two, because an image model and a
text model are different predictors and guessing which you meant is not the
API's job.

In [ ]:
text_model = next(
    (model for model in orinth.models.list() if model.available and model.task_type == "text_classification"),
    None,
)
if text_model:
    answer = orinth.inference.predict(
        text_model.id, text="the battery drains within an hour of charging"
    )
    print(answer.label, "|", answer.text)
else:
    print("No text model in this workspace yet.")

## The history

Everything recorded, newest first — the Inference page's own list, and the
reason it is worth having: you can find last Tuesday's answer without having
kept the notebook open.

In [ ]:
import polars as pl

pl.DataFrame(
    [
        {"id": entry.id[:12], "model": entry.model_id, "label": entry.label, "detections": len(entry.detections)}
        for entry in orinth.inference.list(limit=15)
    ]
)